### Analytical Evaluation of the Effective Dynamics (FAPT)

While the numerical propagation of the effective state is efficient for single pulse sequences, large parameter sweeps and pulse optimizations require a less computationally demanding approach. To achieve this, the perturbative construction can be evaluated analytically.

Instead of performing the time-evolution numerically over many small steps $\delta t$, this approach uses closed-form symbolic expressions for both the transformation matrix $W(\boldsymbol\lambda)$ and the effective Hamiltonian $H_{\mathrm{eff,ad}}$. The time evolution operator $U(t, t_i)$ over the entire pulse duration is then approximated using the **Magnus expansion** (up to second order), effectively evaluating the evolution in a single analytical step. 

The following code tests this analytical framework by substituting symbolic pulse shapes (like a Gaussian amplitude) into the analytically derived Magnus operator.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import sympy as sp
from utils import * 
from FAPT import * 
from pulse_optimization import * 
s = Simulation()
import matplotlib.pyplot as plt
rH = 2
rW = 1

def setup_paper_style():
    """Sets Matplotlib defaults for scientific publication layouts."""
    plt.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Computer Modern", "Times New Roman", "DejaVu Serif"],
        "mathtext.fontset": "cm",
        "text.usetex": False,
        "axes.labelsize": 11.5,
        "font.size": 11,
        "legend.fontsize": 9.0,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "figure.titlesize": 12,
        "axes.titlesize": 11.5,
        "lines.linewidth": 1.4,
        "lines.markersize": 5.0,
    })

In [ ]:
from pathlib import Path
import numpy as np
import sympy as sp
DRIVE_DIRECTORY = Path("operational_res/analytical_drive_data")

def load_analytical_drive(pulse_type, drive_directory=DRIVE_DIRECTORY):
    file_path = Path(drive_directory) / f"analytical_drive_{pulse_type}_data.npz"

    if not file_path.exists():
        raise FileNotFoundError(f"Die Datei '{file_path}' wurde nicht gefunden.")

    with np.load(file_path, allow_pickle=False) as saved_data:
        stored_pulse_type = str(saved_data["pulse_type"].item())
        Ar_expression = sp.sympify(saved_data["Ar_srepr"].item())
        Ai_expression = sp.sympify(saved_data["Ai_srepr"].item())
        dwd_expression = sp.sympify(saved_data["wd_srepr"].item())
    if stored_pulse_type != pulse_type:
        raise ValueError(f"Angefordert wurde '{pulse_type}', die Datei enthält jedoch '{stored_pulse_type}'.")

    pulse_data = {
        "Ar": Ar_expression,
        "Ai": Ai_expression,
        "wd": dwd_expression,
    }
    print(f"Symbolische Pulsform geladen: '{file_path}'")
    return pulse_data

gauss_drive = load_analytical_drive(pulse_type="aa_gauss")
Ar_gauss_expr = gauss_drive["Ar"]
Ai_gauss_expr = gauss_drive["Ai"]
wd_gauss_expr = gauss_drive["wd"]

def get_symbol_by_name(expressions, symbol_name):
    free_symbols = set().union(*(expression.free_symbols for expression in expressions if expression is not None))
    matching_symbols = [symbol for symbol in free_symbols if symbol.name == symbol_name]

    if not matching_symbols:
        raise KeyError(f"Das Symbol '{symbol_name}' kommt in den geladenen Ausdrücken nicht vor.")
    return matching_symbols[0]

loaded_expressions = (Ar_gauss_expr, Ai_gauss_expr, wd_gauss_expr)
t_sym = get_symbol_by_name(expressions=loaded_expressions, symbol_name="t")
amp_scale_sym = get_symbol_by_name(expressions=loaded_expressions, symbol_name="f")
tg_sym = get_symbol_by_name(expressions=loaded_expressions, symbol_name="t_g")

In [ ]:
def replace_loaded_symbols(expression, t_sym, amp_scale_sym, tg_sym):
    symbol_map = {}

    for symbol in expression.free_symbols:
        if symbol.name == "t":
            symbol_map[symbol] = t_sym
        elif symbol.name in ("tg", "t_g", r"t_{g}"):
            symbol_map[symbol] = tg_sym
        elif symbol.name == "f":
            symbol_map[symbol] = amp_scale_sym
    return expression.xreplace(symbol_map)


def make_analytical_pulse_builder(Ar_expr, Ai_expr, wd_expr):
    def pulse_shape_builder(t_sym, p_syms):
        tg_sym, wd_offset_sym, amp_scale_sym = p_syms

        Ar = replace_loaded_symbols(expression=Ar_expr, t_sym=t_sym, amp_scale_sym=amp_scale_sym, tg_sym=tg_sym)
        Ai = replace_loaded_symbols(expression=Ai_expr, t_sym=t_sym, amp_scale_sym=amp_scale_sym, tg_sym=tg_sym)
        wd = replace_loaded_symbols(expression=wd_expr, t_sym=t_sym, amp_scale_sym=amp_scale_sym, tg_sym=tg_sym)

        A = (Ar + sy.I * Ai)
        wd =  wd + wd_offset_sym

        return A, wd

    return pulse_shape_builder

In [ ]:
gauss_pulse_builder = make_analytical_pulse_builder(Ar_expr=Ar_gauss_expr, Ai_expr=Ai_gauss_expr, wd_expr=wd_gauss_expr)

base_parameter_names = [
    "t_g",
    "wd_offset",
    "amp_scale",
]

param_names = ["wd_offset", "amp_scale"]

def create_binder(include_geometric = False, include_micromotion = False, include_g_correction=False, verbose=False, param_names=param_names):
    H_gauss_base, M_gauss_base, M_inv_0_gauss_base = prepare_floquet_functions(pulse_shape_builder=gauss_pulse_builder, p_names=base_parameter_names, s=s, rH=rH, rW=rW, include_geometric=include_geometric, include_micromotion=include_micromotion, include_g_correction=include_g_correction, verbose=verbose)
    def bind_func(tg_target, H_base=H_gauss_base, M_base=M_gauss_base, M_inv_0_base=M_inv_0_gauss_base):
        def absolute_parameters(offsets):
            # Falls offsets als einzelnes Container-Objekt (Tupel/Liste/Array) übergeben wurde, entpacken
            if len(offsets) == 1 and isinstance(offsets[0], (tuple, list, np.ndarray)):
                offsets = offsets[0]
                
            values = dict(zip(param_names, offsets))
            return values["wd_offset"], values["amp_scale"]

        def H_func(t, *offsets):
            wd_offset, amp_scale = absolute_parameters(offsets)
            return H_base(t, tg_target, wd_offset, amp_scale)

        def M_func(t, *offsets):
            wd_offset, amp_scale = absolute_parameters(offsets)
            return M_base(t, tg_target, wd_offset, amp_scale)

        def M_inv_0_func(*offsets):
            wd_offset, amp_scale = absolute_parameters(offsets)
            return M_inv_0_base(tg_target, wd_offset, amp_scale)

        H_func.is_time_dependent = getattr(H_base, "is_time_dependent", True)
        return H_func, M_func, M_inv_0_func
    return bind_func
bind_floquet_functions = create_binder()

In [ ]:
tg_list = np.linspace(50.0, 700.0, 30)
sigma_r = 0.3
param_ranges = {"wd_offset": (-0.005, 0.5), "amp_scale": (0.8, 1.2)}
param_names = list(param_ranges.keys())

def run_tg_sweep(save_name, binder, debug=False, verbose=False, low=False):
    s = Simulation()
    linfid_pop_direct, linfid_pop_floq, linfid_unopt_direct, linfid_unopt_floq = [], [], [], []
    def sim_pulse(tg, amp_scale, wd_offset):
        Ar = sy.lambdify(t_sym, replace_loaded_symbols(Ar_gauss_expr, t_sym,amp_scale, tg), "numpy")
        Ai = sy.lambdify(t_sym, replace_loaded_symbols(Ai_gauss_expr, t_sym, amp_scale, tg), "numpy")
        wd = sy.lambdify(t_sym, replace_loaded_symbols(wd_gauss_expr, t_sym, amp_scale, tg) + wd_offset, "numpy")
        q = lambda t, a=None: Ar(t)*np.cos(wd(t)*t) + Ai(t)*np.sin(wd(t)*t)
        U = np.column_stack([[st.overlap(sesolve([s.H0, [s.V1, q]], s.E_states[s.comp_indices[j]], np.linspace(0, tg, max(101, int(tg*8)))).states[-1]) for st in s.E_states[s.comp_indices]] for j in range(4)])
        return infid_population(U)

    for tg in tqdm(tg_list):
        H_f, M_f, M_i0_f = binder(tg)
        if not low:
            popsize=17 
            maxiter=80
            tol=1e-8
        else:
            popsize=8
            maxiter=40
            tol=1e-5

        p_opt, f_inf = optimize_pulse_parameters(H_f, M_f, M_i0_f, param_ranges, s, tg, tol = tol, popsize = popsize, maxiter = maxiter,x0=np.array([0,1]),verbose=verbose)
        infid_pop_direct = sim_pulse(tg, p_opt["amp_scale"], p_opt["wd_offset"])
        linfid_pop_direct.append(infid_pop_direct)
        if debug:
            print("direct infid: ", infid_pop_direct," inf: ", f_inf, " tg: ", tg, " p_opt: ", p_opt)

        linfid_pop_floq.append(f_inf)
        
        linfid_unopt_direct.append(sim_pulse(tg, 1.0, 0.0))
        
        tlist = np.linspace(0.0, tg, 200)
        dt = tlist[1] - tlist[0]
        U_unopt = compute_propagator_floquet((0.0, 1.0), H_f, M_f, M_i0_f, tlist[:-1] + dt/2.0, dt, tg, s)
        linfid_unopt_floq.append(infid_population(U_unopt))
    Path("operational_res/drag_popfids").mkdir(parents=True, exist_ok=True)
    np.savez(f"operational_res/drag_popfids/drag_popfids_data_{save_name}.npz", tg_list=tg_list, direct=linfid_pop_direct, floq=linfid_pop_floq, unopt_direct=linfid_unopt_direct, unopt_floq=linfid_unopt_floq)

compute for FPT prediction

In [ ]:
run_tg_sweep("custom", bind_floquet_functions, debug=False, verbose=False, low=False)

 73%|███████▎  | 22/30 [18:40<09:11, 68.92s/it]

define plotting methods

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

def load_and_plot(save_name):
    # Daten laden
    ref_data = np.load("operational_res/cos_popfids_data.npz")
    ref_tg_list, ref_infid_pop = ref_data["tg_list"], ref_data["infid_pop_list"]

    drag_data = np.load(f"operational_res/drag_popfids/drag_popfids_data_{save_name}.npz")
    tg_list = drag_data["tg_list"]
    linfid_pop_direct = drag_data["direct"]
    linfid_pop_floq = drag_data["floq"]
    def plot_infidelity_vs_tg_fullwidth(
        tg_list, 
        linfid_pop_direct, 
        linfid_pop_floq, 
        ref_tg_list=None, 
        ref_infid_pop=None, 
        title_suffix="",
        save_path=None, 
        show=True
    ):
        setup_paper_style()
        
        # Farben
        c_direct, c_floq, c_ref = "#1f77b4", "#d62728", "#555555"
        c_unopt_dir, c_unopt_flq = "#17becf", "#ff7f0e"  # Cyan & Orange für unoptimiert

        fullwidth_style = {
            "axes.labelsize": 12.0, "axes.titlesize": 12.5,
            "xtick.labelsize": 10.5, "ytick.labelsize": 10.5,
            "legend.fontsize": 9.5, "lines.linewidth": 1.8, "lines.markersize": 5.5,
        }
        
        with plt.rc_context(fullwidth_style):
            fig, ax = plt.subplots(figsize=(6.8, 3.8), dpi=300)
            
            # 1. Cosine Reference
            if ref_tg_list is not None and ref_infid_pop is not None:
                ax.semilogy(
                    ref_tg_list, ref_infid_pop, ":", color=c_ref, marker="^", 
                    mfc="white", mec=c_ref, mew=1.2, lw=1.5, label=r"Cosine Reference ($I_{\mathrm{pop}}$)"
                )
            # 3. Optimierte Kurven
            ax.semilogy(
                tg_list, linfid_pop_direct, "-", color=c_direct, marker="o", 
                mfc="white", mec=c_direct, mew=1.5, label=r"DRAG Direct ($U_{\mathrm{direct}}$)"
            )
            ax.semilogy(
                tg_list, linfid_pop_floq, "--", color=c_floq, marker="s", 
                mfc="white", mec=c_floq, mew=1.5, label=r"DRAG Floquet ($U_{\mathrm{floq}}$)"
            )
            
            ax.set_xlabel(r"Gate duration $t_g$ [ns]")
            ax.set_ylabel(r"Population Infidelity $I_{\mathrm{pop}}$")
            ax.set_title(rf"Population Infidelity vs. Gate Duration $t_g${title_suffix}", fontweight="bold")
            ax.grid(True, which="both", linestyle=":", alpha=0.5, linewidth=0.8)
            ax.legend(loc="best", frameon=True, fancybox=False, edgecolor="gray", framealpha=0.9)
            
            fig.tight_layout()
            if save_path:
                out = Path(save_path)
                out.parent.mkdir(parents=True, exist_ok=True)
                fig.savefig(out, bbox_inches="tight")
            if show:
                plt.show()
                
            return fig, ax

    # Plot 1: Standard (nur optimierte DRAG-Daten)
    fig1, ax1 = plot_infidelity_vs_tg_fullwidth(
        tg_list, linfid_pop_direct, linfid_pop_floq,
        ref_tg_list=ref_tg_list, ref_infid_pop=ref_infid_pop,
        save_path=f"Figure/drag_results/{save_name}.pdf"
    )

In [ ]:
load_and_plot("custom")

In [ ]:
binder = create_binder(use_micromotion=True)
run_tg_sweep("custom_FAPT")

Check the actual dynamics to argument for FAPT use

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import sesolve

# 1. Parameter & Signal für tg
tg = 100.0
H_f, M_f, M_i0_f = bind_floquet_functions(tg)
p_opt, infid = optimize_pulse_parameters(H_f, M_f, M_i0_f, param_ranges, s, tg, tol=1e-6, popsize=8, x0=np.array([0, 1]), verbose=False)
print("popt: ", p_opt, " infid: ", infid)

Ar = sy.lambdify(t_sym, replace_loaded_symbols(Ar_gauss_expr, t_sym, p_opt["amp_scale"], tg), "numpy")
Ai = sy.lambdify(t_sym, replace_loaded_symbols(Ai_gauss_expr, t_sym, p_opt["amp_scale"], tg), "numpy")
wd = sy.lambdify(t_sym, replace_loaded_symbols(wd_gauss_expr, t_sym, p_opt["amp_scale"], tg) + p_opt["wd_offset"], "numpy")
q = lambda t, a=None: Ar(t)*np.cos(wd(t)*t) + Ai(t)*np.sin(wd(t)*t)

# 2. Dynamics
tlist = np.linspace(0, tg, max(101, int(tg*8)))
res = sesolve([s.H0, [s.V1, q]], s.E_states[s.state_a], tlist)

pop_a = [np.abs(s.E_states[s.state_a].overlap(st))**2 for st in res.states]
pop_b = [np.abs(s.E_states[s.state_b].overlap(st))**2 for st in res.states]
pop_c = [np.abs(s.E_states[s.state_c].overlap(st))**2 for st in res.states]
pop_c[0]=0
# 3. Plot
fig, ax1 = plt.subplots(figsize=(4.5, 2.8), dpi=300)
ax2 = ax1.twinx()

l1, = ax1.plot(tlist/tg, pop_a, "#1f77b4", label=r"$|\langle a|\psi\rangle|^2$")
l2, = ax1.plot(tlist/tg, pop_b, "#ff7f0e", ls="--", label=r"$|\langle b|\psi\rangle|^2$")
l3, = ax2.semilogy(tlist/tg, pop_c, "#d62728", ls="-.", label=r"$|\langle c|\psi\rangle|^2$ (Leakage)")

# Horizontale Linie für die Infidelity auf der rechten Y-Achse
l4 = ax2.axhline(infid, color="gray", ls=":", lw=1.2, label=r"$I_{\mathrm{pop}}$")

ax1.set_xlabel(r"$t/t_{\mathrm{g}}$")
ax1.set_ylabel("Population")
ax2.set_ylabel("Leakage / Infidelity", color="#d62728")
ax2.tick_params(colors="#d62728")

fig.legend(handles=[l1, l2, l3, l4], loc="lower center", bbox_to_anchor=(0.5, 0.92), ncol=4, frameon=False)
fig.savefig("Figure/drag_dynamics.pdf", bbox_inches="tight")
plt.show()
# full: 1.9448873178617987e-08
# without 1.9448862298432346e e-8